# Plots for comparison with the paper

Figures for the replication write-up, covering the roleplay and instructed
scenarios only:

1. **AUROC and recall-at-fixed-FPR by layer** — where in the network deception is
   most linearly decodable.
2. **Regularization sweep** at the best layer.
3. **ROC curve** at the best layer, with the recall-at-1%-FPR operating point
   marked (the paper's headline metric).
4. **Token-level score trajectory** — how the probe's score moves across a single
   response, rather than mean-pooled.

**Inputs:** `data/features_*.npz`, `data/baseline_training_summary.json`,
`data/probe_eval.json`. **Outputs:** PNGs in `plots/`.

## Setup

Shared helpers live in [`common.py`](common.py). Select the `interp` conda
environment as this notebook's kernel — it has `transformer_lens`, `torch`,
`scikit-learn` and `matplotlib` installed.

In [ ]:
import os
import sys
from pathlib import Path

# These notebooks use repo-relative paths ("data/...", "plots/..."), exactly as the
# original scripts did when run from the project root. So make that the working
# directory, and put src/ on the import path so `common` is importable.
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".git").exists()), Path.cwd())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

import common
common.ensure_dirs()
print("Working directory:", Path.cwd())

In [ ]:
import json

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import torch
from scipy.interpolate import interp1d
from sklearn.metrics import roc_auc_score, roc_curve

BEST_LAYER = 17  # best-performing layer from probe_pipeline_main.ipynb

## Load the cached features

`common.load_features` replaces the old `get_training_data` helper — it derives
the available layers from the `.npz` file itself.

In [ ]:
X_train_by_layer, y_train, X_eval_by_layer, y_eval = common.load_features()
layers = sorted(X_train_by_layer)

## 1. AUROC and recall by layer

Recall at a fixed low FPR is the more decision-relevant metric: a monitor has to
run at a false-positive rate an operator can tolerate.

In [ ]:
def recall_at_fpr(clf, X_eval, y_eval, target_fpr=0.01):
    y_scores = clf.predict_proba(X_eval)[:, 1]
    fpr, tpr, _ = roc_curve(y_eval, y_scores)

    # Find the smallest fpr that is >= target_fpr (first threshold meeting the constraint)
    valid = np.where(fpr >= target_fpr)[0]
    if len(valid) == 0:
        idx = len(fpr) - 1  # fallback: highest fpr available
        print(f"No FPR >= {target_fpr:.3f} found; using highest available FPR={fpr[idx]:.3f}")
    else:
        idx = valid[0]

    return tpr[idx], fpr[idx]


def sweep_recall_by_layer(X_train_by_layer, y_train, X_eval_by_layer, y_eval,
                          layers, target_fpr=0.01):
    results = {}
    for layer in layers:
        clf = common.fit_probe(X_train_by_layer[layer], y_train, C=1.0)
        recall, actual_fpr = recall_at_fpr(clf, X_eval_by_layer[layer], y_eval, target_fpr)
        results[layer] = {"recall": recall, "actual_fpr": actual_fpr}
        print(f"Layer {layer}: recall={recall:.3f} at FPR~{actual_fpr:.3f}")
    return results

In [ ]:
def plot_auroc_and_recall(training_summary_path, X_train_by_layer, y_train,
                          X_eval_by_layer, y_eval, target_fpr=0.01):
    with open(training_summary_path) as f:
        training_data = json.load(f)

    results_per_layer = training_data["results_per_layer"]  # {"7": 0.7x, "14": 0.8x, ...}

    layers = sorted(X_train_by_layer)
    aurocs = [results_per_layer[str(l)] for l in layers]

    recall_results = sweep_recall_by_layer(X_train_by_layer, y_train, X_eval_by_layer, y_eval,
                                           layers, target_fpr=target_fpr)
    recalls_pct = [recall_results[l]["recall"] * 100 for l in layers]
    returned_fpr = recall_results[layers[0]]["actual_fpr"]

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(7, 8), sharex=True,
        gridspec_kw={"height_ratios": [1, 1], "hspace": 0.1}
    )

    # Top panel — AUROC
    ax1.plot(layers, aurocs, marker='o', linewidth=2, markersize=8, color="#2b6cb0")
    ax1.set_ylabel("AUROC")
    ax1.set_ylim(0.45, 1.05)
    ax1.grid(alpha=0.3)
    ax1.set_title("AUROC and Recall by Layer")

    # Bottom panel — recall at fixed FPR
    ax2.plot(layers, recalls_pct, marker='s', linewidth=2, markersize=8, color="#c0392b")
    ax2.set_ylabel(f"Recall @ ~{returned_fpr*100:.0f}% FPR")
    ax2.set_ylim(0, 105)
    ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax2.grid(alpha=0.3)

    ax2.set_xlabel("Probe layer")
    ax2.set_xticks(layers)

    plt.tight_layout()
    plt.savefig("plots/auroc_recall_stacked.png", dpi=150)
    return recall_results

In [ ]:
recall_results = plot_auroc_and_recall(
    "data/baseline_training_summary.json",
    X_train_by_layer, y_train, X_eval_by_layer, y_eval,
)

## 2. Regularization sweep at the best layer

In [ ]:
def sweep_regularization(X_train, y_train, X_eval, y_eval, C_values):
    results = {}
    for C in C_values:
        clf = common.fit_probe(X_train, y_train, C=C)
        y_scores = clf.predict_proba(X_eval)[:, 1]
        auroc = roc_auc_score(y_eval, y_scores)
        results[C] = auroc
        print(f"C={C:.4f} (lambda~{1/C:.2f}): AUROC={auroc:.3f}")
    return results


def plot_regularization_sweep(X_train_by_layer, y_train, X_eval_by_layer, y_eval, layer):
    C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
    reg_results = sweep_regularization(
        X_train_by_layer[layer], y_train, X_eval_by_layer[layer], y_eval, C_values
    )
    C_vals = sorted(reg_results.keys())
    aurocs = [reg_results[c] for c in C_vals]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(C_vals, aurocs, marker='o', linewidth=2, markersize=8, color="#2b6cb0")
    ax.set_xscale('log')
    ax.set_xlabel("Regularization Coefficient (= 1/lambda, log scale)")
    ax.set_ylabel("AUROC")
    ax.set_title(f"AUROC vs. Regularization — Layer {layer}" if layer else "AUROC vs. Regularization")
    ax.set_ylim(0.9, 1.05)
    ax.grid(alpha=0.3, which='both')
    plt.tight_layout()
    plt.savefig(f"plots/regularization_sweep_layer{layer}.png", dpi=150)
    return reg_results

In [ ]:
reg_results = plot_regularization_sweep(
    X_train_by_layer, y_train, X_eval_by_layer, y_eval, layer=BEST_LAYER
)

## 3. ROC curve with the recall-at-1%-FPR operating point

In [ ]:
def plot_roc_with_recall_at_fpr(clf, X_eval, y_eval, target_fpr=0.01, layer=None, smooth=True):
    recall_at_target, actual_fpr = recall_at_fpr(clf, X_eval, y_eval, target_fpr)

    fig, ax = plt.subplots(figsize=(6, 6))

    y_scores = clf.predict_proba(X_eval)[:, 1]
    fpr, tpr, _ = roc_curve(y_eval, y_scores)
    auroc = roc_auc_score(y_eval, y_scores)
    fpr_pct = fpr * 100
    tpr_pct = tpr * 100

    if smooth:
        # Deduplicate fpr for interpolation (roc_curve can have repeated fpr values)
        fpr_unique, idx = np.unique(fpr_pct, return_index=True)
        tpr_unique = tpr_pct[idx]

        interp_fn = interp1d(fpr_unique, tpr_unique, kind='linear')
        fpr_smooth = np.linspace(fpr_unique.min(), fpr_unique.max(), 100)
        tpr_smooth = interp_fn(fpr_smooth)

        ax.plot(fpr_smooth, tpr_smooth, linewidth=2, label=f"Interpolated ROC (AUROC={auroc:.3f})")
    else:
        ax.plot(fpr_pct, tpr_pct, linewidth=2, label=f"ROC (AUROC={auroc:.3f})")

    ax.scatter([actual_fpr], [recall_at_target], color="red", zorder=5,
               label=f"Recall={recall_at_target:.2f} @ FPR~{actual_fpr:.3f}")
    ax.plot([0, 100], [0, 100], linestyle="--", color="lightgray", linewidth=1, label="Chance")

    ax.set_xlim(0, 100)
    ax.set_ylim(0, 110)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_xlabel("FPR on Honest")
    ax.set_ylabel("Recall (TPR)")
    ax.axvline(1, color="red", linestyle=":", linewidth=1, alpha=0.7, label="1% FPR")
    ax.set_title(f"ROC Curve — Layer {layer}" if layer is not None else "ROC Curve")
    ax.legend(loc="lower right", fontsize=9)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"plots/roc_curve_layer{layer}.png", dpi=150)

    print(f"AUROC: {auroc:.3f}")
    print(f"At {actual_fpr:.3f} FPR (target {target_fpr}), recall = {recall_at_target:.3f}")
    return auroc, recall_at_target

In [ ]:
clf = common.fit_probe(X_train_by_layer[BEST_LAYER], y_train, C=1.0)

auroc, recall = plot_roc_with_recall_at_fpr(
    clf, X_eval_by_layer[BEST_LAYER], y_eval, layer=BEST_LAYER
)

## 4. Token-level probe score trajectory

This one needs the model, since it scores every response token individually
rather than reading the mean-pooled cache.

Pinned to CPU/bfloat16, matching how the cached features were extracted.

In [ ]:
model = common.load_model(device="cpu", dtype=torch.bfloat16)

In [ ]:
def get_token_level_scores(model, prompt_text, response_text, clf, layer):
    """
    Returns the probe's per-token deception score across the response,
    rather than a single mean-pooled score.
    """
    full_text = prompt_text + response_text
    tokens = model.to_tokens(full_text)
    prompt_tokens = model.to_tokens(prompt_text)
    response_start = prompt_tokens.shape[1]

    with torch.no_grad():
        _, cache = model.run_with_cache(
            tokens,
            names_filter=lambda name: name == f"blocks.{layer}.hook_resid_post"
        )

    resid = cache[f"blocks.{layer}.hook_resid_post"][0]          # [seq_len, d_model]
    response_resid = resid[response_start:].float().numpy()       # [n_response_tokens, d_model]

    # Score every token individually using the trained probe
    token_scores = clf.predict_proba(response_resid)[:, 1]        # P(deceptive) per token

    response_token_ids = tokens[0, response_start:]
    response_token_strs = [model.to_string(t.unsqueeze(0)) for t in response_token_ids]

    return response_token_strs, token_scores


def plot_token_trajectory(model, eval_data_path, clf, layer, example_idx=0):
    with open(eval_data_path) as f:
        eval_data = json.load(f)

    example = eval_data[example_idx]
    label = f"{example['label']}_example{example_idx}"

    print(f"Obtaining token-level scores for {label} (layer {layer})...")
    tokens_str, scores = get_token_level_scores(
        model, example["prompt"], example["response"], clf, layer
    )

    plt.figure(figsize=(10, 3))
    plt.plot(scores, marker='o', markersize=3)
    plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
    plt.xlabel("Token position in response")
    plt.ylabel("P(deceptive)")
    plt.title(f"Token-level probe score — {label}")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(f"plots/token_trajectory_{label.replace(' ', '_')}.png", dpi=150)

    return tokens_str, scores

In [ ]:
tokens_str, token_scores = plot_token_trajectory(
    model, "data/probe_eval.json", clf=clf, layer=BEST_LAYER
)

Per-token scores for a quick manual read:

In [ ]:
for tok, score in zip(tokens_str, token_scores):
    print(f"{score:.2f}  {tok!r}")